In [1]:
import json
from sklearn.metrics import cohen_kappa_score
# Round 1
annotated_ann1 = "data_ann1.json"
annotated_ann2 = "data_ann2.json"
with open(annotated_ann1, 'r', encoding='utf-8') as file:
    ann1 = json.load(file)


with open(annotated_ann2, 'r', encoding='utf-8') as file:
    ann2 = json.load(file)

In [2]:
TYPES = ["Lexical", "Syntactic", "Semantic", "Vagueness", "Incompleteness", "Referential"]

# ann1gn by id instead of trusting file order
ann1_by_id = {r["id"]: r for r in ann1}
ann2_by_id = {r["id"]: r for r in ann2}
ids = sorted(set(ann1_by_id) & set(ann2_by_id))
print(f"{len(ids)} shared ids "
      f"(ann1 only: {sorted(set(ann1_by_id) - set(ann2_by_id))}, "
      f"ann2 only: {sorted(set(ann2_by_id) - set(ann1_by_id))})")

for t in TYPES:
    a = [int(ann1_by_id[i][t]) for i in ids]
    w = [int(ann2_by_id[i][t]) for i in ids]
    agree = sum(x == y for x, y in zip(a, w)) / len(ids)
    if len(set(a)) == 1 and len(set(w)) == 1 and a[0] == w[0]:
        k = float("nan")  # both annotators used one label only — kappa undefined
    else:
        k = cohen_kappa_score(a, w, labels=[0, 1])
    print(f"{t:<15} kappa={k:6.3f}  agreement={agree:.2%}  "
          f"ann1_pos={sum(a):2d}  ann2_pos={sum(w):2d}")

148 shared ids (ann1 only: [], ann2 only: [])
Lexical         kappa= 1.000  agreement=100.00%  ann1_pos= 7  ann2_pos= 7
Syntactic       kappa= 0.832  agreement=97.97%  ann1_pos= 8  ann2_pos=11
Semantic        kappa= 1.000  agreement=100.00%  ann1_pos= 1  ann2_pos= 1
Vagueness       kappa= 0.941  agreement=97.30%  ann1_pos=53  ann2_pos=53
Incompleteness  kappa= 0.860  agreement=95.95%  ann1_pos=26  ann2_pos=26
Referential     kappa= 0.554  agreement=95.95%  ann1_pos=10  ann2_pos= 4


In [3]:
disagreements = {t: [i for i in ids if ann1_by_id[i][t] != ann2_by_id[i][t]] for t in TYPES}
disagreements

{'Lexical': [],
 'Syntactic': [111, 112, 117],
 'Semantic': [],
 'Vagueness': [3, 4, 136, 137],
 'Incompleteness': [4, 6, 7, 8, 136, 137],
 'Referential': [8, 107, 108, 109, 110, 113]}

In [4]:
for i in ids:
    diffs = [t for t in TYPES if bool(ann1_by_id[i][t]) != bool(ann2_by_id[i][t])]
    if not diffs:
        continue
    print(f"id: {i}")
    print(f"requirement: {ann1_by_id[i]['description']}")
    for t in diffs:
        who = "ann1" if ann1_by_id[i][t] else "ann2"
        print(f"  - {t}: True by {who}")
    for name, rec in (("ann1", ann1_by_id[i]), ("ann2", ann2_by_id[i])):
        note = (rec.get("notes") or rec.get("note") or "").strip()
        if note:
            print(f"  note ({name}): {note}")
    print()

id: 3
requirement: The system shall implement procedures for changing passwords.
  - Vagueness: True by ann1

id: 4
requirement: The system shall implement procedures for safeguarding passwords.
  - Vagueness: True by ann2
  - Incompleteness: True by ann1

id: 6
requirement: The system shall identify and respond to suspected or known security incidents.
  - Incompleteness: True by ann2

id: 7
requirement: The system shall mitigate harmful effects to the extent practicable after a suspected or known security incidents.
  - Incompleteness: True by ann2

id: 8
requirement: The system shall document security incidents and their outcomes.
  - Incompleteness: True by ann1
  - Referential: True by ann1

id: 107
requirement: The system must ensure that the notice of privacy practices informs individuals of their right to inspect their protected health information.
  - Referential: True by ann1
  note (ann1): second their is not clear that is mentioning to individual or the system

id: 108
requ